# Transactions

A transaction commits all changes together or rolls them all back.

In [ ]:
import sqlite3

db = sqlite3.connect(":memory:")
db.execute("CREATE TABLE accounts (id INTEGER PRIMARY KEY, balance INTEGER)")
db.executemany("INSERT INTO accounts VALUES (?, ?)", [(1, 100), (2, 50)])
db.commit()

try:
    with db:
        db.execute("UPDATE accounts SET balance = balance - 30 WHERE id = 1")
        db.execute("UPDATE accounts SET balance = balance + 30 WHERE id = 2")
        raise RuntimeError("simulated failure")
except RuntimeError:
    pass

print(db.execute("SELECT * FROM accounts").fetchall())

The balances remain unchanged because the transaction rolled back.

## Polished version

A standalone service defines the business rule while one transaction commits both account updates or rolls both back.

In [ ]:
import sqlite3


class AccountRepository:
    def __init__(self, connection: sqlite3.Connection) -> None:
        self.connection = connection

    def balance(self, account_id: int) -> int:
        row = self.connection.execute(
            "SELECT balance FROM accounts WHERE id = ?",
            (account_id,),
        ).fetchone()
        if row is None:
            raise ValueError("account not found")
        return int(row[0])

    def change_balance(self, account_id: int, amount: int) -> None:
        self.connection.execute(
            "UPDATE accounts SET balance = balance + ? WHERE id = ?",
            (amount, account_id),
        )


class TransferService:
    def __init__(self, connection: sqlite3.Connection) -> None:
        self.connection = connection
        self.accounts = AccountRepository(connection)

    def transfer(self, sender_id: int, receiver_id: int, amount: int) -> None:
        if amount <= 0:
            raise ValueError("amount must be positive")

        with self.connection:
            if self.accounts.balance(sender_id) < amount:
                raise ValueError("insufficient balance")
            self.accounts.change_balance(sender_id, -amount)
            self.accounts.change_balance(receiver_id, amount)


connection = sqlite3.connect(":memory:")
connection.execute("CREATE TABLE accounts (id INTEGER PRIMARY KEY, balance INTEGER)")
connection.executemany("INSERT INTO accounts VALUES (?, ?)", [(1, 100), (2, 50)])
connection.commit()

transfers = TransferService(connection)
try:
    transfers.transfer(1, 2, 500)
except ValueError:
    pass

transfers.transfer(1, 2, 20)
print(connection.execute("SELECT * FROM accounts ORDER BY id").fetchall())

## Applied in this repository

The REST [session dependency](../00P1-project-rest-api/app/database.py) commits after a successful request and rolls back on exceptions; routers only flush the work they add.